# Day 33: LLMOps
**The LLM lifecycle, AgentOps, resilience, fine-tuning, versioning, and cost and latency**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Install

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai tiktoken scikit-learn numpy pandas matplotlib

# ===== Option B: VS Code / local =====
# pip install openai tiktoken scikit-learn numpy pandas matplotlib
# Create a .env file (git-ignored) or set the variable in your terminal:  OPENAI_API_KEY=...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

def count_tokens(text):
    try:
        import tiktoken
        return len(tiktoken.get_encoding("o200k_base").encode(text))
    except Exception:
        return max(1, round(len(text) / 4))          # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

## 2. Incident simulation: a week of LLM request logs
Planted events: a rate-limit storm on 2 Dec and a prompt change that triples the context from 4 Dec.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
rng = np.random.default_rng(31)
n = 8000
ts = pd.Timestamp("2026-11-30") + pd.to_timedelta(rng.integers(0, 7 * 24 * 60, n), unit="min")
logs = pd.DataFrame({"ts": ts, "model": "main-model-v1", "status": 200,
                     "latency_ms": rng.gamma(4, 300, n).round(), "tokens": rng.normal(900, 150, n).round()})
day, hour = logs.ts.dt.date.astype(str), logs.ts.dt.hour
burst = (day == "2026-12-02") & (hour == 10)                            # rate limit storm
logs.loc[burst & (rng.random(n) < 0.55), "status"] = 429
logs.loc[rng.random(n) < 0.01, "status"] = 503                          # background provider errors
logs.loc[logs.ts >= "2026-12-04", "tokens"] *= 3                        # a prompt change added a huge context
logs.loc[logs.status != 200, "latency_ms"] = rng.integers(20000, 30000, (logs.status != 200).sum())
logs = logs.sort_values("ts").reset_index(drop=True)

by_hour = logs.set_index("ts").resample("1h").agg(errors=("status", lambda s: (s != 200).mean()))
worst = by_hour.errors.idxmax()
print(f"Worst hour: {worst} with {by_hour.errors.max():.0%} errors")
print("Status codes in that hour:",
      logs[(logs.ts >= worst) & (logs.ts < worst + pd.Timedelta("1h"))].status.value_counts().to_dict())
daily_tokens = logs.groupby(logs.ts.dt.date)["tokens"].mean().round()
print("\nAverage tokens per request by day:\n", daily_tokens.to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
by_hour.errors.plot(ax=ax[0], color="#FF8A3D", title="Error rate per hour")
daily_tokens.plot(ax=ax[1], kind="bar", color="#0FA3A3", title="Average tokens per request")
plt.tight_layout(); plt.show()

## 3. A flaky provider: retries with backoff and jitter
The simulated provider fails 30% of the time with transient errors. Waits are scaled down so the demo runs fast.

In [ ]:
import random, time

class Transient(Exception):
    pass

class FlakyProvider:
    def __init__(self, fail_rate, name):
        self.fail_rate, self.name, self.calls = fail_rate, name, 0
    def complete(self, prompt):
        self.calls += 1
        if random.random() < self.fail_rate:
            raise Transient(f"{self.name}: 429 or timeout")
        return f"[{self.name}] answer to: {prompt[:30]}"

def with_retry(fn, retries=3, base=0.001):
    for attempt in range(retries + 1):
        try:
            return fn()
        except Transient:
            if attempt == retries:
                raise
            time.sleep(base * (2 ** attempt) + random.random() * base)    # exponential backoff with jitter

random.seed(31)
main = FlakyProvider(0.30, "main")
for retries in [0, 2, 4]:
    ok = 0
    for i in range(500):
        try:
            with_retry(lambda: main.complete("hello"), retries=retries); ok += 1
        except Transient:
            pass
    print(f"retries={retries}: success {ok / 500:.0%}")

## 4. Fallback model and circuit breaker

In [ ]:
class CircuitBreaker:
    def __init__(self, threshold=3, cooldown=0.05):
        self.threshold, self.cooldown = threshold, cooldown
        self.failures, self.opened_at, self.state = 0, None, "CLOSED"
        self.log = ["CLOSED"]

    def _set(self, state):
        if state != self.state:
            self.state = state
            self.log.append(state)

    def call(self, fn):
        if self.state == "OPEN":
            if time.time() - self.opened_at < self.cooldown:
                raise Transient("circuit open")
            self._set("HALF-OPEN")
        try:
            result = fn()
            self.failures = 0
            self._set("CLOSED")
            return result
        except Transient:
            self.failures += 1
            if self.failures >= self.threshold or self.state == "HALF-OPEN":
                self._set("OPEN")
                self.opened_at = time.time()
            raise

random.seed(7)
outage = FlakyProvider(0.95, "main")          # the main provider is having an outage
backup = FlakyProvider(0.02, "backup")
breaker = CircuitBreaker()
served = {"main": 0, "backup": 0, "error message": 0}
for i in range(40):
    time.sleep(0.005)                          # requests arrive over time, so the cool-down can expire
    try:
        breaker.call(lambda: outage.complete("Where is my order?")); served["main"] += 1
    except Transient:
        try:
            with_retry(lambda: backup.complete("Where is my order?"), retries=1); served["backup"] += 1
        except Transient:
            served["error message"] += 1
print("Served by:", served)
print("Calls that reached the failing main provider:", outage.calls, "out of 40 requests")
print("Breaker transitions:", " -> ".join(breaker.log[:9]), "...")

**TODO 1:** make the circuit breaker log every state change with a timestamp, and add a `served_from_cache` option for frequent questions.

In [ ]:
# TODO: write your code here

## 5. NEW (v4): AgentOps, a release manifest for the whole agent
An agent is prompts, tools, skills, MCP servers and a model. Hash each part so any silent change shows up.

In [ ]:
import hashlib, json, copy

def h(obj):
    text = obj if isinstance(obj, str) else json.dumps(obj, sort_keys=True)
    return hashlib.sha256(text.encode()).hexdigest()[:6]

agent = {
    "prompts": {"system": "You are SmartDesk, a telecom support agent. Cite policy ids. Never promise refunds over 5,000 JPY."},
    "tools": {"refund_tool": {"name": "refund", "description": "Issue a refund up to 5,000 JPY",
                              "parameters": {"ticket_id": "int", "amount_jpy": "int"}},
              "lookup_ticket": {"name": "lookup_ticket", "description": "Read a ticket by id", "parameters": {"ticket_id": "int"}}},
    "skills": {"triage": "# SKILL.md\nClassify tickets as billing, network or account before answering."},
    "mcp_servers": {"tickets": "2.1.0"},
    "model": {"name": MODEL, "reasoning_effort": "medium"},   # model name from LLM_MODEL; see docs/MODEL_GUIDE.md
}

def manifest(agent, release):
    return {"release": release,
            "prompts": {k: h(v) for k, v in agent["prompts"].items()},
            "tools": {k: h(v) for k, v in agent["tools"].items()},
            "skills": {k: h(v) for k, v in agent["skills"].items()},
            "mcp_servers": dict(agent["mcp_servers"]),
            "model": dict(agent["model"]),
            "golden_set": "golden_v9"}

def changed_parts(a, b):
    out = []
    for part in ["prompts", "tools", "skills", "mcp_servers", "model"]:
        for k in set(a[part]) | set(b[part]):
            if a[part].get(k) != b[part].get(k):
                out.append(f"{part}.{k}")
    return out

m1 = manifest(agent, "smartdesk-agent-2.3.0")
agent2 = copy.deepcopy(agent)
agent2["tools"]["refund_tool"]["description"] = "Issue a refund up to 10,000 JPY"     # a risky silent change
m2 = manifest(agent2, "smartdesk-agent-2.3.1")
print(json.dumps(m1, indent=2))
print("changed parts:", changed_parts(m1, m2))

**Why it matters:** the tool change above doubles the refund limit without touching the prompt. Only a per-part hash makes that visible in review.

## 6. NEW (v4): Model deprecation runbook drill
A provider announces a retirement. Find every affected release, evaluate a candidate on the golden set, then decide. Example from 2026: GPT-5.5 retired from Codex for ChatGPT sign-ins on 14 Oct 2026. Confirm candidates against the approved model list.

In [ ]:
manifests = [
    {"release": "smartdesk-agent-2.3.0", "model": "legacy-model-a"},
    {"release": "faq-bot-1.1.0", "model": "legacy-model-a"},
    {"release": "summariser-0.9.0", "model": MODEL},
]
notice = {"retiring": "legacy-model-a", "date": "2027-01-31"}
candidate = MODEL                                          # confirm against docs/MODEL_GUIDE.md

affected = [m["release"] for m in manifests if m["model"] == notice["retiring"]]
print("1. Watch    : notice for", notice["retiring"], "retiring on", notice["date"])
print("2. Inventory: affected releases", affected)

GOLDEN = [("Classify this ticket: I was charged twice for my plan", "billing"),
          ("Classify this ticket: No signal at home since Monday", "network"),
          ("Classify this ticket: Cannot log in after password reset", "account"),
          ("Classify this ticket: Refund for duplicate invoice", "billing"),
          ("Classify this ticket: 5G data is very slow downtown", "network")]
passed = sum(llm("Reply with one word: billing, network or account.\n" + q, model=candidate).strip().lower() == want
             for q, want in GOLDEN)
rate = passed / len(GOLDEN)
print(f"3. Evaluate : candidate {candidate} golden pass rate {rate:.2f} (offline scores show the code path only)")
decision = "CANARY" if rate >= 0.9 else "HOLD: fix prompts or try another approved model"
print("4. Canary   :", decision)
print("5. Switch   : update manifests, keep previous manifests for rollback until", notice["date"])
print("6. Record   : decision log entry written")

## 7. LLMOps vs MLOps: preparing fine-tuning data

In [ ]:
import re
random.seed(32)
templates = {
    "billing": ["I was charged twice for order {o}", "Refund for {o} not received", "Invoice amount is wrong, contact me at {e}"],
    "network": ["No signal at home near {o}", "Data is very slow today", "Calls drop all the time, call me on {p}"],
    "account": ["Cannot log in to my account", "Change the plan on {o}", "SIM swap request, email {e}"],
}
rows = []
for label, ts_ in templates.items():
    for _ in range(20):
        t = random.choice(ts_).format(o=f"ORD-{random.randint(1000, 9999)}", e="mei@example.com", p="+81 90 1234 5678")
        rows.append({"text": t, "label": label})

PII = [r"[\w.+-]+@[\w-]+\.[\w.]+", r"\+?\d{1,3}[ -]?\d{2,5}[ -]?\d{3,4}[ -]?\d{3,4}"]
def scrub(t):
    for p in PII:
        t = re.sub(p, "[redacted]", t)
    return t

SYSTEM = "Classify the support ticket as billing, network or account. Reply with one word."
examples, pii_removed = [], 0
for r in rows:
    clean = scrub(r["text"]); pii_removed += clean != r["text"]
    examples.append({"messages": [{"role": "system", "content": SYSTEM},
                                  {"role": "user", "content": clean},
                                  {"role": "assistant", "content": r["label"]}]})

def valid(ex):
    m = ex["messages"]
    return ([x["role"] for x in m] == ["system", "user", "assistant"] and all(x["content"].strip() for x in m)
            and m[2]["content"] in templates)

random.shuffle(examples)
split = int(len(examples) * 0.8)
for name, part in [("train.jsonl", examples[:split]), ("val.jsonl", examples[split:])]:
    with open(name, "w") as f:
        for ex in part:
            f.write(json.dumps(ex) + "\n")
print(f"Valid: {sum(map(valid, examples))}/{len(examples)} | PII removed: {pii_removed} | train {split}, validation {len(examples) - split}")
print(examples[0])

In [ ]:
import os
RUN_FINE_TUNE = False     # costs money: set True only with trainer approval, an API key and an approved fine-tunable model
if ONLINE and RUN_FINE_TUNE:
    import genai_helpers
    c = genai_helpers.client
    train_file = c.files.create(file=open("train.jsonl", "rb"), purpose="fine-tune")
    val_file = c.files.create(file=open("val.jsonl", "rb"), purpose="fine-tune")
    job = c.fine_tuning.jobs.create(training_file=train_file.id, validation_file=val_file.id,
                                    model=os.environ["FT_BASE_MODEL"])    # set from the approved model list
    print("Started fine-tuning job:", job.id)
else:
    print("Fine-tuning not started (RUN_FINE_TUNE is False). The data files are ready: train.jsonl, val.jsonl")

## 8. A prompt registry and release manifest

In [ ]:
import difflib, os
registry = {}
def register(pid, text, note):
    registry[pid] = {"text": text, "sha": h(text), "note": note}
    return registry[pid]["sha"]

register("answer_v7", "Answer using ONLY the context.\nCite chunk ids.\nIf unsure, say I don't know.", "baseline")
register("answer_v8", "Answer using ONLY the context.\nCite chunk ids.\nAnswer in at most 3 sentences.\nIf unsure, say I don't know.",
         "shorter answers to cut output tokens")
for line in difflib.unified_diff(registry["answer_v7"]["text"].splitlines(), registry["answer_v8"]["text"].splitlines(),
                                 "answer_v7", "answer_v8", lineterm=""):
    print(line)

## 9. Cost and latency from request logs
**The prices below are placeholders, not real prices.** Replace them with the approved price sheet. This builds on the Day 25 token cost calculator.

In [ ]:
PRICES = {"large-model": {"in": 2.50, "out": 10.00}, "small-model": {"in": 0.15, "out": 0.60}}   # PLACEHOLDER per 1M tokens

rng = np.random.default_rng(32)
n = 6000
feature = rng.choice(["faq_bot", "summariser", "agent_research"], n, p=[0.7, 0.21, 0.09])
model = np.where(feature == "faq_bot", rng.choice(["large-model", "small-model"], n, p=[0.5, 0.5]), "large-model")
tok_in = np.where(feature == "agent_research", rng.normal(14000, 3000, n),
                  np.where(feature == "summariser", rng.normal(3000, 600, n), rng.normal(800, 150, n))).clip(100)
tok_out = np.where(feature == "summariser", rng.normal(400, 80, n), rng.normal(180, 50, n)).clip(10)
lat = (np.where(model == "large-model", 900, 300) + tok_out * np.where(model == "large-model", 12, 4) + rng.gamma(2, 150, n))
req = pd.DataFrame({"feature": feature, "model": model, "tok_in": tok_in.round(), "tok_out": tok_out.round(),
                    "latency_ms": lat.round()})
req["cost"] = [(r.tok_in * PRICES[r.model]["in"] + r.tok_out * PRICES[r.model]["out"]) / 1e6 for r in req.itertuples()]

by_feature = req.groupby("feature").agg(requests=("cost", "size"), cost=("cost", "sum"))
by_feature["share_of_requests"] = (by_feature.requests / n).round(2)
by_feature["share_of_cost"] = (by_feature.cost / by_feature.cost.sum()).round(2)
print(by_feature.round(2).to_string(), "\n")
print("Latency by model (ms):\n", req.groupby("model").latency_ms.quantile([0.5, 0.95]).unstack().round().to_string())

faq = req[req.feature == "faq_bot"]
for m in PRICES:
    c = ((faq.tok_in * PRICES[m]["in"] + faq.tok_out * PRICES[m]["out"]) / 1e6).sum()
    print(f"FAQ bot all on {m:<12}: {c:.2f} (placeholder units)")

## 10. Caching and routing: latency vs accuracy
Quality numbers here are **simulated** to show the trade-off. Measure real quality on your golden set.

In [ ]:
questions = [f"faq question {i}" for i in range(3000)]
weights = 1 / np.arange(1, 3001) ** 1.1; weights /= weights.sum()          # a few questions are very common
stream = rng.choice(questions, 5000, p=weights)
cache, hits = {}, 0
for q in stream:
    key = q.strip().lower()
    if key in cache:
        hits += 1
    else:
        cache[key] = f"answer to {q}"
print(f"Cache hit rate {hits / len(stream):.0%} | model calls {len(stream) - hits} instead of {len(stream)}")

qs = pd.DataFrame({"hard": rng.random(2000) < 0.3})
QUALITY = {("large-model", True): 0.90, ("large-model", False): 0.93, ("small-model", True): 0.70, ("small-model", False): 0.92}
def run(policy):
    m = qs.hard.map(policy)
    quality = np.mean([QUALITY[(mm, hh)] for mm, hh in zip(m, qs.hard)])
    cost = sum((800 * PRICES[mm]["in"] + 180 * PRICES[mm]["out"]) / 1e6 for mm in m)
    return quality, cost
_, base_c = run(lambda hard: "large-model")
for name, pol in [("all large", lambda hard: "large-model"), ("all small", lambda hard: "small-model"),
                  ("routed", lambda hard: "large-model" if hard else "small-model")]:
    q, c = run(pol)
    print(f"{name:<10} quality {q:.3f} | cost {c / base_c:.0%} of all-large (simulated quality, placeholder prices)")

**TODO 2 (SmartDesk step 28):** write a simple difficulty classifier (length, number of questions, words like "compare" or "why") and use it instead of the `hard` column. Log the release manifest id with every request.

In [ ]:
# TODO: write your code here

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)